# EV Charging Locations × SA4 Spatial Join


## Purpose


This notebook integrates the cleaned EV charging location dataset with the NSW ABS SA4 spatial boundaries.

The objective is to assign each EV charging location to its corresponding SA4 region using a geospatial point-in-polygon join.

Pipeline:

EV Silver
+
ABS SA4 Silver
↓
Load and validate datasets
↓
Convert EV coordinates to point geometries
↓
Align coordinate reference systems
↓
Spatial join EV points to SA4 polygons
↓
Validate matched and unmatched locations
↓
Create enriched EV–SA4 Silver dataset

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd


PROJECT_ROOT = Path.cwd().parent

EV_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "ev"
    / "ev_charging_locations_silver.parquet"
)

ABS_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "abs"
    / "nsw_sa4_silver.parquet"
)

print("Project root:", PROJECT_ROOT)

print("\nEV Silver:")
print(EV_SILVER_PATH)
print("Exists:", EV_SILVER_PATH.exists())

print("\nABS Silver:")
print(ABS_SILVER_PATH)
print("Exists:", ABS_SILVER_PATH.exists())

Project root: D:\mycode\nsw-ev-data-pipeline

EV Silver:
D:\mycode\nsw-ev-data-pipeline\data\silver\ev\ev_charging_locations_silver.parquet
Exists: True

ABS Silver:
D:\mycode\nsw-ev-data-pipeline\data\silver\abs\nsw_sa4_silver.parquet
Exists: True


## Step 2 — Load the Silver Datasets


The EV Silver dataset contains cleaned charging-location attributes.

The ABS Silver dataset contains validated NSW SA4 polygon geometries.

Before performing any spatial operation, both datasets are inspected to confirm their schema and spatial metadata.

In [2]:
ev_silver = pd.read_parquet(EV_SILVER_PATH)
sa4_silver = gpd.read_parquet(ABS_SILVER_PATH)

print("EV Silver shape:", ev_silver.shape)
print("ABS Silver shape:", sa4_silver.shape)

print("\nEV columns:")
print(ev_silver.columns.tolist())

print("\nABS columns:")
print(sa4_silver.columns.tolist())

print("\nABS CRS:")
print(sa4_silver.crs)

EV Silver shape: (1958, 33)
ABS Silver shape: (28, 8)

EV columns:
['source_object_id', 'operator_raw', 'charger_type_raw', 'charger_rating_raw', 'lga_name_raw', 'postcode_raw', 'source', 'station_name', 'station_address', 'operator', 'operator_standardised', 'number_of_plugs', 'latitude', 'longitude', 'lga_name', 'postcode', 'charger_type', 'is_upcoming', 'address_postcode', 'charger_power_kw', 'charger_power_min_kw', 'charger_power_max_kw', 'coordinate_record_count', 'address_needs_review', 'postcode_conflict', 'charger_rating_is_complex', 'charger_rating_needs_review', 'number_of_plugs_invalid', 'number_of_plugs_high_outlier', 'coordinate_invalid', 'coordinate_direction_suspicious', 'repeated_coordinate', 'potential_duplicate']

ABS columns:
['SA4_CODE26', 'SA4_NAME26', 'GCC_CODE26', 'GCC_NAME26', 'STE_CODE26', 'STE_NAME26', 'AREASQKM26', 'geometry']

ABS CRS:
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "GDA2020", "datum": {"typ

In [3]:
coordinate_candidates = [
    column
    for column in ev_silver.columns
    if any(
        keyword in column.lower()
        for keyword in ["lat", "lon", "lng", "coord"]
    )
]

print("Possible coordinate columns:")
print(coordinate_candidates)

Possible coordinate columns:
['latitude', 'longitude', 'coordinate_record_count', 'coordinate_invalid', 'coordinate_direction_suspicious', 'repeated_coordinate']


## Step 3 — Validate EV Coordinates

Before creating spatial point geometries, the cleaned latitude and longitude fields are inspected.

The validation confirms:

- coordinate data types;
- missing coordinate values;
- coordinate ranges;
- previously generated coordinate quality flags.

This reduces the risk of assigning an incorrect coordinate reference system to the EV locations.

In [4]:
print("Coordinate dtypes:")
print(ev_silver[["latitude", "longitude"]].dtypes)

print("\nMissing coordinates:")
print(ev_silver[["latitude", "longitude"]].isna().sum())

print("\nCoordinate summary:")
print(
    ev_silver[
        ["latitude", "longitude"]
    ].describe()
)

print("\nCoordinate quality flags:")
print("Invalid:", ev_silver["coordinate_invalid"].sum())
print(
    "Direction suspicious:",
    ev_silver["coordinate_direction_suspicious"].sum()
)

Coordinate dtypes:
latitude     float64
longitude    float64
dtype: object

Missing coordinates:
latitude     0
longitude    0
dtype: int64

Coordinate summary:
          latitude    longitude
count  1958.000000  1958.000000
mean    -33.484300   150.697229
std       1.588800     1.665905
min     -37.111463   141.460104
25%     -33.944959   150.486904
50%     -33.853122   151.156083
75%     -33.010212   151.266295
max     -28.168593   153.615875

Coordinate quality flags:
Invalid: 0
Direction suspicious: 0


## Step 4 — Convert EV Locations to Spatial Points


The EV charging locations are converted from tabular longitude and latitude values into point geometries.

The source coordinates are interpreted as WGS 84 (`EPSG:4326`), which is the standard geographic CRS commonly used for longitude and latitude coordinates.

The resulting GeoDataFrame will subsequently be transformed to the CRS used by the ABS SA4 boundaries.

In [5]:
ev_points = gpd.GeoDataFrame(
    ev_silver.copy(),
    geometry=gpd.points_from_xy(
        ev_silver["longitude"],
        ev_silver["latitude"],
    ),
    crs="EPSG:4326",
)

print("Type:", type(ev_points))
print("Shape:", ev_points.shape)
print("CRS:", ev_points.crs)

print("\nMissing geometries:")
print(ev_points.geometry.isna().sum())

print("\nEmpty geometries:")
print(ev_points.geometry.is_empty.sum())

print("\nGeometry types:")
print(ev_points.geom_type.value_counts())

Type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (1958, 34)
CRS: EPSG:4326

Missing geometries:
0

Empty geometries:
0

Geometry types:
Point    1958
Name: count, dtype: int64


## Step 5 — Align Coordinate Reference Systems


Spatial joins require both spatial datasets to use a compatible coordinate reference system.

The EV point geometries are therefore transformed from WGS 84 (`EPSG:4326`) to the GDA2020 CRS (`EPSG:7844`) used by the ABS SA4 boundaries.

In [6]:
ev_points_7844 = ev_points.to_crs(sa4_silver.crs)

print("EV CRS:", ev_points_7844.crs.to_epsg())
print("SA4 CRS:", sa4_silver.crs.to_epsg())

print(
    "CRS aligned:",
    ev_points_7844.crs == sa4_silver.crs,
)

EV CRS: 7844
SA4 CRS: 7844
CRS aligned: True


## Step 6 — Spatially Join EV Locations to SA4 Regions

A point-in-polygon spatial join is performed to assign each EV charging location to the SA4 region containing it.

The EV charging locations are used as the left dataset so that all charging locations are retained, including any locations that fail to match an SA4 polygon.

In [7]:
sa4_join_columns = [
    "SA4_CODE26",
    "SA4_NAME26",
    "GCC_CODE26",
    "GCC_NAME26",
    "AREASQKM26",
    "geometry",
]

ev_sa4_joined = gpd.sjoin(
    ev_points_7844,
    sa4_silver[sa4_join_columns],
    how="left",
    predicate="within",
)

print("Original EV rows:", len(ev_points_7844))
print("Joined rows:", len(ev_sa4_joined))

print(
    "Matched:",
    ev_sa4_joined["SA4_CODE26"].notna().sum(),
)

print(
    "Unmatched:",
    ev_sa4_joined["SA4_CODE26"].isna().sum(),
)

Original EV rows: 1958
Joined rows: 1958
Matched: 1957
Unmatched: 1


In [8]:
matched = ev_sa4_joined["SA4_CODE26"].notna().sum()
total = len(ev_sa4_joined)

match_rate = matched / total

print(
    f"Spatial join match rate: "
    f"{match_rate:.2%}"
)

Spatial join match rate: 99.95%


## Step 7 — Investigate Unmatched EV Locations


The initial point-in-polygon spatial join matched 1,957 of 1,958 EV charging locations, producing a 99.95% match rate.

The single unmatched record is investigated before applying any fallback logic.

Possible causes include:

- a point located exactly on an SA4 boundary;
- a coordinate slightly outside the NSW polygon coverage;
- a coastal or offshore location;
- a source-data coordinate issue.

In [10]:
unmatched_ev = ev_sa4_joined[
    ev_sa4_joined["SA4_CODE26"].isna()
].copy()

unmatched_columns = [
    "source_object_id",
    "station_name",
    "station_address",
    "operator_standardised",
    "lga_name",
    "postcode",
    "latitude",
    "longitude",
]

unmatched_ev[unmatched_columns]

,source_object_id,station_name,station_address,operator_standardised,lga_name,postcode,latitude,longitude
1833,<NA>,<NA>,1 Sandy Bay Rd Clontarf NSW 2093 Australia,EVX,Northern Beaches Council,2093,-33.80467,151.25284


In [11]:
unmatched_points = ev_points_7844.loc[
    unmatched_ev.index
].copy()

boundary_check = gpd.sjoin(
    unmatched_points,
    sa4_silver[
        [
            "SA4_CODE26",
            "SA4_NAME26",
            "GCC_NAME26",
            "geometry",
        ]
    ],
    how="left",
    predicate="intersects",
)

boundary_check[
    [
        "station_name",
        "latitude",
        "longitude",
        "SA4_CODE26",
        "SA4_NAME26",
        "GCC_NAME26",
    ]
]

,station_name,latitude,longitude,SA4_CODE26,SA4_NAME26,GCC_NAME26
1833,<NA>,-33.80467,151.25284,NaN,NaN,NaN


### Nearest-SA4 Distance Check

Because the unmatched EV location does not intersect any SA4 polygon, the distance to the nearest SA4 boundary is calculated.

A projected coordinate reference system is used so that the distance can be interpreted in metres.

This helps distinguish a small coastal/geocoding offset from a genuinely incorrect location.

In [13]:
unmatched_projected = unmatched_points.to_crs("EPSG:9473")
sa4_projected = sa4_silver.to_crs("EPSG:9473")

nearest = gpd.sjoin_nearest(
    unmatched_projected,
    sa4_projected[
        [
            "SA4_CODE26",
            "SA4_NAME26",
            "GCC_CODE26",
            "GCC_NAME26",
            "geometry",
        ]
    ],
    how="left",
    distance_col="distance_to_sa4_m",
)

nearest[
    [
        "station_address",
        "latitude",
        "longitude",
        "SA4_CODE26",
        "SA4_NAME26",
        "GCC_NAME26",
        "distance_to_sa4_m",
    ]
]

,station_address,latitude,longitude,SA4_CODE26,SA4_NAME26,GCC_NAME26,distance_to_sa4_m
1833,1 Sandy Bay Rd Clontarf NSW 2093 Australia,-33.80467,151.25284,122,Sydney - Northern Beaches,Greater Sydney,1.755318


In [14]:
print(
    "Nearest SA4:",
    nearest["SA4_NAME26"].iloc[0],
)

print(
    "Distance to nearest SA4:",
    round(nearest["distance_to_sa4_m"].iloc[0], 2),
    "metres",
)

Nearest SA4: Sydney - Northern Beaches
Distance to nearest SA4: 1.76 metres


## Step 8 — Resolve the Remaining Unmatched Location


The initial point-in-polygon join matched 1,957 of 1,958 EV locations.

The remaining location at 1 Sandy Bay Rd, Clontarf was only 1.76 metres from the Sydney - Northern Beaches SA4 polygon.

This is consistent with a small coastal-boundary or coordinate precision difference rather than an incorrect location.

A conservative nearest-region fallback is therefore applied only to unmatched locations within 10 metres of an SA4 polygon.

The match method and fallback distance are retained to preserve spatial data lineage.

In [15]:
import pandas as pd


FALLBACK_MAX_DISTANCE_M = 10.0

ev_sa4_final = ev_sa4_joined.copy()

# Record how each SA4 assignment was obtained.
ev_sa4_final["sa4_match_method"] = pd.Series(
    "within",
    index=ev_sa4_final.index,
    dtype="string",
)

ev_sa4_final["sa4_match_distance_m"] = pd.Series(
    pd.NA,
    index=ev_sa4_final.index,
    dtype="Float64",
)

# Identify nearest matches that satisfy the conservative tolerance.
fallback_matches = nearest[
    nearest["distance_to_sa4_m"] <= FALLBACK_MAX_DISTANCE_M
].copy()

fallback_matches[
    [
        "SA4_CODE26",
        "SA4_NAME26",
        "GCC_CODE26",
        "GCC_NAME26",
        "distance_to_sa4_m",
    ]
]

,SA4_CODE26,SA4_NAME26,GCC_CODE26,GCC_NAME26,distance_to_sa4_m
1833,122,Sydney - Northern Beaches,1GSYD,Greater Sydney,1.755318


## Step 8 — Apply Controlled Nearest-SA4 Fallback


The initial point-in-polygon join matched 1,957 of 1,958 EV charging locations.

The remaining location was only 1.76 metres from the Sydney - Northern Beaches SA4 polygon.

A conservative 10-metre nearest-region fallback is applied only to unmatched records. The matching method and fallback distance are retained to preserve data lineage.

In [17]:
FALLBACK_MAX_DISTANCE_M = 10.0

ev_sa4_final = ev_sa4_joined.copy()

# Track how each SA4 assignment was produced.
ev_sa4_final["sa4_match_method"] = pd.Series(
    pd.NA,
    index=ev_sa4_final.index,
    dtype="string",
)

ev_sa4_final.loc[
    ev_sa4_final["SA4_CODE26"].notna(),
    "sa4_match_method",
] = "within"

ev_sa4_final["sa4_match_distance_m"] = pd.Series(
    pd.NA,
    index=ev_sa4_final.index,
    dtype="Float64",
)

# Keep only unmatched records close enough to an SA4 polygon.
fallback_matches = nearest[
    nearest["distance_to_sa4_m"] <= FALLBACK_MAX_DISTANCE_M
].copy()

print("Fallback candidates:", len(fallback_matches))

Fallback candidates: 1


In [18]:
fallback_columns = [
    "SA4_CODE26",
    "SA4_NAME26",
    "GCC_CODE26",
    "GCC_NAME26",
]

for column in fallback_columns:
    ev_sa4_final.loc[
        fallback_matches.index,
        column,
    ] = fallback_matches[column]

ev_sa4_final.loc[
    fallback_matches.index,
    "sa4_match_method",
] = "nearest_fallback"

ev_sa4_final.loc[
    fallback_matches.index,
    "sa4_match_distance_m",
] = fallback_matches["distance_to_sa4_m"]

In [19]:
area_lookup = sa4_silver.set_index("SA4_CODE26")["AREASQKM26"]

fallback_index = (
    ev_sa4_final["sa4_match_method"] == "nearest_fallback"
)

ev_sa4_final.loc[
    fallback_index,
    "AREASQKM26",
] = (
    ev_sa4_final.loc[
        fallback_index,
        "SA4_CODE26",
    ]
    .map(area_lookup)
)

In [20]:
total = len(ev_sa4_final)

matched = ev_sa4_final["SA4_CODE26"].notna().sum()
unmatched = ev_sa4_final["SA4_CODE26"].isna().sum()

print("Total EV locations:", total)
print("Matched:", matched)
print("Unmatched:", unmatched)
print(f"Final match rate: {matched / total:.2%}")

print("\nMatch methods:")
print(
    ev_sa4_final["sa4_match_method"]
    .value_counts(dropna=False)
)

print("\nFallback records:")
display(
    ev_sa4_final.loc[
        ev_sa4_final["sa4_match_method"] == "nearest_fallback",
        [
            "station_address",
            "latitude",
            "longitude",
            "SA4_CODE26",
            "SA4_NAME26",
            "GCC_NAME26",
            "AREASQKM26",
            "sa4_match_method",
            "sa4_match_distance_m",
        ],
    ]
)

Total EV locations: 1958
Matched: 1958
Unmatched: 0
Final match rate: 100.00%

Match methods:
sa4_match_method
within              1957
nearest_fallback       1
Name: count, dtype: int64[pyarrow]

Fallback records:


,station_address,latitude,longitude,SA4_CODE26,SA4_NAME26,GCC_NAME26,AREASQKM26,sa4_match_method,sa4_match_distance_m
1833,1 Sandy Bay Rd Clontarf NSW 2093 Australia,-33.80467,151.25284,122,Sydney - Northern Beaches,Greater Sydney,254.2073,nearest_fallback,1.755318


## Step 9 — Finalise the Enriched EV–SA4 Silver Dataset


The spatially enriched dataset is cleaned before persistence.

The ABS source field names are standardised for downstream analytics, temporary spatial-join fields are removed, and the final point geometry is stored in WGS 84 (`EPSG:4326`) for interoperability with mapping and analytics tools.

The original EV records are retained without row loss.

In [21]:
ev_sa4_silver = ev_sa4_final.copy()

# Remove temporary GeoPandas spatial-join metadata.
if "index_right" in ev_sa4_silver.columns:
    ev_sa4_silver = ev_sa4_silver.drop(columns=["index_right"])

# Standardise geographic enrichment field names.
ev_sa4_silver = ev_sa4_silver.rename(
    columns={
        "SA4_CODE26": "sa4_code",
        "SA4_NAME26": "sa4_name",
        "GCC_CODE26": "gcc_code",
        "GCC_NAME26": "gcc_name",
        "AREASQKM26": "sa4_area_sqkm",
    }
)

# Store EV point geometries in WGS84 for interoperability.
ev_sa4_silver = ev_sa4_silver.to_crs("EPSG:4326")

print("Type:", type(ev_sa4_silver))
print("Shape:", ev_sa4_silver.shape)
print("CRS:", ev_sa4_silver.crs.to_epsg())

print("\nSA4 enrichment columns:")
print(
    ev_sa4_silver[
        [
            "sa4_code",
            "sa4_name",
            "gcc_code",
            "gcc_name",
            "sa4_area_sqkm",
            "sa4_match_method",
            "sa4_match_distance_m",
        ]
    ].head()
)

Type: <class 'geopandas.geodataframe.GeoDataFrame'>
Shape: (1958, 41)
CRS: 4326

SA4 enrichment columns:
  sa4_code                           sa4_name gcc_code        gcc_name  \
0      106        Hunter Valley exc Newcastle    1RNSW     Rest of NSW   
1      116                 Sydney - Blacktown    1GSYD  Greater Sydney   
2      110         New England and North West    1RNSW     Rest of NSW   
3      121  Sydney - North Sydney and Hornsby    1GSYD  Greater Sydney   
4      112                   Richmond - Tweed    1RNSW     Rest of NSW   

   sa4_area_sqkm sa4_match_method  sa4_match_distance_m  
0     21491.2923           within                  <NA>  
1       240.8836           within                  <NA>  
2     99139.9014           within                  <NA>  
3       275.0992           within                  <NA>  
4     10271.3204           within                  <NA>  


In [22]:
assert len(ev_sa4_silver) == len(ev_silver)
assert ev_sa4_silver.index.is_unique

assert ev_sa4_silver["sa4_code"].notna().all()
assert ev_sa4_silver["sa4_name"].notna().all()
assert ev_sa4_silver["gcc_code"].notna().all()
assert ev_sa4_silver["gcc_name"].notna().all()
assert ev_sa4_silver["sa4_area_sqkm"].notna().all()

assert ev_sa4_silver.geometry.notna().all()
assert ev_sa4_silver.geometry.is_valid.all()
assert (ev_sa4_silver.geom_type == "Point").all()

assert ev_sa4_silver.crs.to_epsg() == 4326

assert (
    ev_sa4_silver["sa4_match_method"]
    .value_counts()
    .to_dict()
    == {
        "within": 1957,
        "nearest_fallback": 1,
    }
)

fallback_mask = (
    ev_sa4_silver["sa4_match_method"]
    == "nearest_fallback"
)

assert (
    ev_sa4_silver.loc[
        fallback_mask,
        "sa4_match_distance_m",
    ]
    <= FALLBACK_MAX_DISTANCE_M
).all()

print("Final EV-SA4 Silver validation passed.")

Final EV-SA4 Silver validation passed.


In [23]:
EV_SA4_SILVER_DIR = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "ev_sa4"
)

EV_SA4_SILVER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EV_SA4_SILVER_PATH = (
    EV_SA4_SILVER_DIR
    / "ev_charging_locations_sa4_silver.parquet"
)

ev_sa4_silver.to_parquet(
    EV_SA4_SILVER_PATH,
    index=False,
)

print("Saved to:", EV_SA4_SILVER_PATH)
print("Exists:", EV_SA4_SILVER_PATH.exists())
print("Size:", EV_SA4_SILVER_PATH.stat().st_size, "bytes")

Saved to: D:\mycode\nsw-ev-data-pipeline\data\silver\ev_sa4\ev_charging_locations_sa4_silver.parquet
Exists: True
Size: 190198 bytes


In [24]:
ev_sa4_roundtrip = gpd.read_parquet(
    EV_SA4_SILVER_PATH
)

assert ev_sa4_roundtrip.shape == ev_sa4_silver.shape

assert (
    ev_sa4_roundtrip.columns.tolist()
    == ev_sa4_silver.columns.tolist()
)

assert (
    ev_sa4_roundtrip.crs
    == ev_sa4_silver.crs
)

assert (
    ev_sa4_roundtrip["sa4_code"].tolist()
    == ev_sa4_silver["sa4_code"].tolist()
)

assert (
    ev_sa4_roundtrip.geometry.to_wkb().tolist()
    == ev_sa4_silver.geometry.to_wkb().tolist()
)

assert (
    ev_sa4_roundtrip["sa4_match_method"]
    .value_counts()
    .to_dict()
    == {
        "within": 1957,
        "nearest_fallback": 1,
    }
)

print("EV-SA4 Silver round-trip validation passed.")

EV-SA4 Silver round-trip validation passed.
